# 05 — AutoFE Search and Final Validation

Финальный AutoFE-эксперимент на постановке с реалистичными отрицательными кандидатами: автоматический поиск новых исторических признаков, калибровка и независимая проверка найденных улучшений.

> Этот ноутбук содержит подготовительный negatives-evaluation, после которого выполняется основной AutoFE-поиск.


In [ ]:
import os, glob, json, time, gc, warnings
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import lightgbm as lgb

warnings.filterwarnings("ignore", message="Overriding the parameters")

SEED, N_USERS = 42, 1500
DAY, SHIFT, TRAIN_DAYS = 86400, 26, 60
SEARCH_ROOTS = ["/kaggle/input", "/kaggle/working", "."]

def find_file(name, required=True):
    for root in SEARCH_ROOTS:
        hits = glob.glob(os.path.join(root, "**", name), recursive=True)
        if hits:
            return hits[0]
    if required:
        raise FileNotFoundError(
            f"не найден файл {name}: подключите Output предыдущей версии через Add Input"
        )
    return None

os.makedirs("results", exist_ok=True)

BASE = find_file("base_listens.parquet")
df = pd.read_parquet(BASE)

print("df:", df.shape, "| исходные колонки:", list(df.columns))

df["pr"] = (np.minimum(df["played_ratio_pct"], 100) / 100).astype(np.float32)
df["skip"] = (df["pr"] < 0.1).astype(np.float32)
df["full"] = (df["pr"] >= 0.9).astype(np.float32)

t_val, t_test = df["timestamp"].quantile([0.8, 0.9])
chk = np.where(
    df["timestamp"] < t_val,
    "train",
    np.where(df["timestamp"] < t_test, "val", "test")
)
assert (chk == df["split"].to_numpy()).all(), "разбиение не совпадает с сохранённым"

# ВАЖНО: negatives_eval использует artist_id и album_id.
# В исходном ноутбуке они восстанавливаются из mapping-файлов.
def load_map(fname):
    p = find_file(fname, required=False)
    if p is None:
        from huggingface_hub import hf_hub_download
        p = hf_hub_download(
            repo_id="yandex/yambda",
            repo_type="dataset",
            filename=fname,
            local_dir="data"
        )
    m = pq.read_table(p).to_pandas()
    other = [c for c in m.columns if c != "item_id"]
    assert len(other) == 1
    return m.drop_duplicates("item_id").set_index("item_id")[other[0]]

for col, fname_ in [
    ("artist_id", "artist_item_mapping.parquet"),
    ("album_id", "album_item_mapping.parquet"),
]:
    s = df["item_id"].map(load_map(fname_)).astype("float64")
    df[col] = np.where(
        s.isna(),
        -1 - df["item_id"].astype(np.int64),
        s
    ).astype(np.int64)

print(
    "артистов/альбомов:",
    df["artist_id"].nunique(),
    df["album_id"].nunique()
)

# Те же train/val и user-level sel/rep, что в исходном ноутбуке.
tr = (
    (df["split"] == "train")
    & (df["timestamp"] >= t_val - TRAIN_DAYS * DAY)
)
va = df["split"] == "val"

rs = np.random.default_rng(123)
val_users = df.loc[va, "uid"].unique()
half = set(
    rs.choice(
        val_users,
        size=len(val_users) // 2,
        replace=False
    ).tolist()
)
is_sel = df["uid"].isin(half)

print(
    "train:", int(tr.sum()),
    "| val:", int(va.sum()),
    "| sel:", int((va & is_sel).sum()),
    "| rep:", int((va & ~is_sel).sum())
)


## 1. Загрузка исходных данных и подготовка минимального состояния


In [ ]:
BASE = find_file("base_listens.parquet")
df = pd.read_parquet(BASE)
print("df:", df.shape)
print("колонки:", list(df.columns))

# Эти поля нужны для совместимости с negatives_eval.py.
df["pr"] = (np.minimum(df["played_ratio_pct"], 100) / 100).astype(np.float32)
df["skip"] = (df["pr"] < 0.1).astype(np.float32)
df["full"] = (df["pr"] >= 0.9).astype(np.float32)

t_val = df["timestamp"].quantile(0.8)
t_test = df["timestamp"].quantile(0.9)
chk = np.where(df["timestamp"] < t_val, "train", np.where(df["timestamp"] < t_test, "val", "test"))
assert (chk == df["split"].to_numpy()).all(), "разбиение не совпадает с сохранённым"

tr = (df["split"] == "train") & (df["timestamp"] >= t_val - TRAIN_DAYS * DAY)
va = df["split"] == "val"

rs = np.random.default_rng(123)
val_users = df.loc[va, "uid"].unique()
half = set(rs.choice(val_users, size=len(val_users) // 2, replace=False).tolist())
is_sel = df["uid"].isin(half)

print("train:", int(tr.sum()), "| val:", int(va.sum()))
print("sel пользователей:", len(half), "| rep пользователей:", len(val_users) - len(half))

## 2. `negatives_eval.py`

Ниже сохранена реализация из исходного ноутбука. Она сама строит признаки, добавляет непрослушанные кандидаты и обучает LightGBM.

In [ ]:
# =====================================================================
# negatives_eval.py  --  приоритет 3: реалистичное множество кандидатов
#
# Проблема (раздел 8 отчёта): кандидаты = только прослушанные треки, поэтому NDCG
# измеряет ранжирование ВНУТРИ прослушанного. Здесь к каждой группе (пользователь, день)
# добавляются n_neg непрослушанных за этот день треков (половина равномерно из каталога,
# половина пропорционально популярности), и ручной набор, полный пул и RFE сравниваются
# на одинаковых кандидатах по NDCG@10 и Recall@10.
#
# Самодостаточный модуль (не требует глобалов ноутбука). Проверен на синтетике
# (наивный перебор для признаков, инварианты негативов, сквозной запуск).
#
# Использование в ноутбуке 3 (после того как есть df, tr, va, is_sel, t_val):
#   from negatives_eval import run_negatives_experiment
#   res = run_negatives_experiment(df, tr.to_numpy(), (va & is_sel).to_numpy(), (va & ~is_sel).to_numpy(), n_neg=20)
# =====================================================================
import time, gc
import numpy as np, pandas as pd, lightgbm as lgb

DAY, SHIFT = 86400, 26
FAMILIES = {"user": ["uid"], "ui": ["uid", "item_id"], "uart": ["uid", "artist_id"], "ualb": ["uid", "album_id"],
            "item": ["item_id"], "art": ["artist_id"], "alb": ["album_id"]}
TARGETS = {"user": ("label", "pr"), "ui": ("label", "pr"), "uart": ("label", "pr"), "ualb": ("label", "pr"),
           "item": ("label",), "art": ("label",), "alb": ("label",)}
WINDOWS = (1, 7, 30, 90, 400)
TAG = {"label": "rate", "pr": "mpr", "skip": "skiprate", "full": "fullrate"}
LGB = dict(objective="lambdarank", metric="ndcg", eval_at=[10], learning_rate=0.05, num_leaves=63,
           min_data_in_leaf=200, feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1, verbosity=-1, num_threads=4)


# ---------------------------------------------------------------------------
# 1. Признаки «запрос против событий»: для строк q (в т.ч. синтетических негативов)
#    агрегаты по РЕАЛЬНЫМ событиям ev строго до начала суток запроса.
# ---------------------------------------------------------------------------
def query_window_feats(ev, q, keys, windows, targets, name, snap=DAY, shift=SHIFT):
    n = len(ev)
    codes = pd.concat([ev[keys], q[keys]], ignore_index=True).groupby(keys, sort=False).ngroup().to_numpy(np.int64)
    ce, cq = codes[:n], codes[n:]
    te = ev["timestamp"].to_numpy(np.int64)
    comb_e = (ce << shift) | te
    o = np.argsort(comb_e, kind="stable"); comb_s = comb_e[o]
    tq = (q["timestamp"].to_numpy(np.int64) // snap) * snap
    hi = np.searchsorted(comb_s, (cq << shift) | tq, side="left")
    cums = {t: np.concatenate([[0.0], np.cumsum(ev[t].to_numpy(np.float64)[o])]) for t in targets}
    out = {}
    for w in windows:
        lo = np.searchsorted(comb_s, (cq << shift) | np.maximum(tq - w * snap, 0), side="left")
        cnt = (hi - lo).astype(np.float64)
        out[f"{name}_cnt_{w}d"] = cnt.astype(np.float32)
        for t in targets:
            out[f"{name}_{TAG[t]}_{w}d"] = np.where(cnt > 0, (cums[t][hi] - cums[t][lo]) / np.maximum(cnt, 1), np.nan).astype(np.float32)
    return out


def build_features(ev, q, families=FAMILIES, targets=TARGETS, windows=WINDOWS):
    out = {}
    for name, keys in families.items():
        out.update(query_window_feats(ev, q, keys, windows, targets[name], name))
    return pd.DataFrame(out, index=q.index)


# ---------------------------------------------------------------------------
# 2. Негативы
# ---------------------------------------------------------------------------
def sample_negatives(pos, ev, catalog_end, n_neg, seed=0, day=DAY):
    """pos: позитивные строки сплита (uid, timestamp, item_id) с колонкой g.
    Каталог и популярность берутся только по событиям до catalog_end (без будущего).
    Негатив не должен совпадать ни с одним треком, прослушанным этим пользователем в этот день."""
    r = np.random.default_rng(seed)
    past = ev[ev["timestamp"] < catalog_end]
    cat, cnt = np.unique(past["item_id"].to_numpy(), return_counts=True)
    p_pop = cnt / cnt.sum()
    groups = np.unique(pos["g"].to_numpy()); G = len(groups); m = 2 * n_neg
    half = m // 2
    draw = np.empty((G, m), dtype=np.int64)
    draw[:, :half] = cat[r.integers(0, len(cat), (G, half))]
    draw[:, half:] = cat[r.choice(len(cat), (G, m - half), p=p_pop)]
    gg = np.repeat(groups, m); it = draw.ravel()
    M = int(max(ev["item_id"].max(), it.max())) + 1
    key_pos = pos["g"].to_numpy(np.int64) * M + pos["item_id"].to_numpy(np.int64)
    key = gg.astype(np.int64) * M + it
    keep = ~np.isin(key, key_pos)
    tab = pd.DataFrame({"g": gg[keep], "item_id": it[keep], "key": key[keep]}).drop_duplicates("key")
    tab = tab[tab.groupby("g").cumcount() < n_neg]
    attr = ev.drop_duplicates("item_id").set_index("item_id")[["artist_id", "album_id"]]
    tab = tab.join(attr, on="item_id")
    tab = tab.dropna(subset=["artist_id"])
    neg = pd.DataFrame({"uid": tab["g"].to_numpy() // 1000, "timestamp": (tab["g"].to_numpy() % 1000) * day + day // 2,
                        "item_id": tab["item_id"].to_numpy(), "artist_id": tab["artist_id"].to_numpy().astype(np.int64),
                        "album_id": tab["album_id"].to_numpy().astype(np.int64), "label": 0.0, "g": tab["g"].to_numpy(), "is_neg": 1})
    return neg


def build_split(df, mask, n_neg, seed, min_pos_rows=10):
    cols = ["uid", "timestamp", "item_id", "artist_id", "album_id", "label"]
    pos = df.loc[mask, cols].copy()
    pos["g"] = pos["uid"].astype(np.int64) * 1000 + pos["timestamp"].astype(np.int64) // DAY
    pos["is_neg"] = 0
    catalog_end = int(pos["timestamp"].min())
    neg = sample_negatives(pos, df, catalog_end, n_neg, seed)
    rows = pd.concat([pos, neg], ignore_index=True).sort_values("g", kind="stable").reset_index(drop=True)
    npos_rows = pos.groupby("g").size()
    rows["elig"] = rows["g"].map(npos_rows).ge(min_pos_rows).to_numpy()          # группы, как в исходном протоколе
    F = build_features(df, rows)
    return rows, F


# ---------------------------------------------------------------------------
# 3. Метрики и бутстрап
# ---------------------------------------------------------------------------
def group_metrics(g, s, y, k=10):
    d = pd.DataFrame({"g": np.asarray(g), "s": np.asarray(s, np.float64), "y": np.asarray(y, np.float64)})
    a = d.sort_values(["g", "s"], ascending=[True, False], kind="stable")
    ra = a.groupby("g").cumcount().to_numpy() + 1
    dcg = pd.Series(a["y"].to_numpy() / np.log2(ra + 1) * (ra <= k), index=a.index).groupby(a["g"]).sum()
    hits = pd.Series(a["y"].to_numpy() * (ra <= k), index=a.index).groupby(a["g"]).sum()
    b = d.sort_values(["g", "y"], ascending=[True, False], kind="stable")
    rb = b.groupby("g").cumcount().to_numpy() + 1
    idcg = pd.Series(b["y"].to_numpy() / np.log2(rb + 1) * (rb <= k), index=b.index).groupby(b["g"]).sum()
    npos = d.groupby("g")["y"].sum()
    out = pd.DataFrame({"ndcg": dcg / idcg, "recall": hits / npos})
    return out[npos > 0]


def boot_ci(m_a, m_b, col, n_boot=1000, seed=0):
    """Кластерный bootstrap по пользователям для разности (b - a) по группам."""
    diff = (m_b[col] - m_a[col]).dropna()
    pu = diff.groupby(diff.index // 1000).agg(["sum", "count"])
    sums, cnts = pu["sum"].to_numpy(), pu["count"].to_numpy()
    r = np.random.default_rng(seed); est = []
    for _ in range(n_boot):
        i = r.integers(0, len(sums), len(sums)); est.append(sums[i].sum() / cnts[i].sum())
    return float(diff.mean()), (float(np.percentile(est, 2.5)), float(np.percentile(est, 97.5)))


# ---------------------------------------------------------------------------
# 4. Модели
# ---------------------------------------------------------------------------
def _ds(rows, F, cols, ref=None):
    sz = rows.groupby("g", sort=True).size().to_numpy()
    return lgb.Dataset(F[cols].to_numpy(np.float32), label=rows["label"].to_numpy(), group=sz,
                       params={"max_bin": 63}, reference=ref, free_raw_data=False)


def fit_rank(R_tr, F_tr, R_sel, F_sel, cols, seed=0, rounds=300):
    dtr = _ds(R_tr, F_tr, cols); dsel = _ds(R_sel, F_sel, cols, dtr)
    m = lgb.train({**LGB, "seed": seed}, dtr, rounds, valid_sets=[dsel],
                  callbacks=[lgb.early_stopping(30, verbose=False), lgb.log_evaluation(0)])
    return m


def predict(m, F, cols):
    return m.predict(F[cols].to_numpy(np.float32), num_iteration=m.best_iteration or None)


def rfe_select(R_tr, F_tr, R_sel, F_sel, cols, K, drop=0.15, seed=0):
    cols = list(cols)
    while len(cols) > K:
        m = fit_rank(R_tr, F_tr, R_sel, F_sel, cols, seed=seed, rounds=220)
        imp = pd.Series(m.feature_importance("gain"), index=cols).sort_values(ascending=False)
        cols = imp.index[:max(K, int(len(cols) * (1 - drop)))].tolist()
    return cols


MANUAL_WINDOWS = (1, 7, 30)
def manual_cols(all_cols):
    keep = []
    for fam in ("user", "ui", "item"):
        for w in MANUAL_WINDOWS:
            keep += [f"{fam}_cnt_{w}d", f"{fam}_rate_{w}d"]
    return [c for c in keep if c in all_cols]


def run_negatives_experiment(df, mask_tr, mask_sel, mask_rep, n_neg=20, seed=0, K_rfe=None, out_csv="results/negatives_eval.csv"):
    """df: таблица прослушиваний с колонками uid, timestamp, item_id, artist_id, album_id, label, pr (как в ноутбуке).
    mask_*: булевы массивы по строкам df (train-окно, sel, rep). Возвращает таблицу метрик."""
    t0 = time.time()
    df = df[["uid", "timestamp", "item_id", "artist_id", "album_id", "label", "pr"]].reset_index(drop=True)
    splits = {}
    for nm, mk in [("tr", mask_tr), ("sel", mask_sel), ("rep", mask_rep)]:
        splits[nm] = build_split(df, np.asarray(mk), n_neg, seed + {"tr": 1, "sel": 2, "rep": 3}[nm])
        R_, F_ = splits[nm]
        print(f"[{nm}] строк {len(R_)}, негативов {int(R_['is_neg'].sum())}, групп {R_['g'].nunique()}, признаков {F_.shape[1]} | {round(time.time()-t0)} с", flush=True)
    (R_tr, F_tr), (R_sel, F_sel), (R_rep, F_rep) = splits["tr"], splits["sel"], splits["rep"]
    all_cols = list(F_tr.columns); man = manual_cols(all_cols); K = K_rfe or len(man)
    models = {"manual": man, "pool": all_cols}
    models[f"rfe{K}"] = rfe_select(R_tr, F_tr, R_sel, F_sel, all_cols, K, seed=seed)
    el = R_rep["elig"].to_numpy()
    pop_score = F_rep["item_cnt_30d"].fillna(0).to_numpy()
    mets = {"random": group_metrics(R_rep.loc[el, "g"], np.random.default_rng(0).random(el.sum()), R_rep.loc[el, "label"]),
            "popularity": group_metrics(R_rep.loc[el, "g"], pop_score[el], R_rep.loc[el, "label"])}
    for nm, cols in models.items():
        m = fit_rank(R_tr, F_tr, R_sel, F_sel, cols, seed=seed)
        mets[nm] = group_metrics(R_rep.loc[el, "g"], predict(m, F_rep.loc[el], cols), R_rep.loc[el, "label"])
        print(f"[{nm}] {len(cols)} признаков | NDCG@10 {mets[nm]['ndcg'].mean():.4f} | Recall@10 {mets[nm]['recall'].mean():.4f}", flush=True)
    rows = [dict(модель=k, признаков=len(models.get(k, [])), ndcg10=v["ndcg"].mean(), recall10=v["recall"].mean(), групп=len(v)) for k, v in mets.items()]
    for a, b in [("manual", "pool"), ("manual", f"rfe{K}"), ("pool", f"rfe{K}")]:
        for col in ("ndcg", "recall"):
            d_, ci = boot_ci(mets[a], mets[b], col)
            rows.append(dict(модель=f"{b} − {a}", метрика=col, d=d_, lo=ci[0], hi=ci[1]))
            print(f"  {b} − {a} [{col}]: {d_:+.4f} [{ci[0]:+.4f}; {ci[1]:+.4f}]")
    res = pd.DataFrame(rows)
    res.to_csv(out_csv, index=False)
    return res


In [ ]:
# Проверяем, есть ли нужные колонки
print("artist_id:", "artist_id" in df.columns)
print("album_id:", "album_id" in df.columns)

if "artist_id" not in df.columns or "album_id" not in df.columns:
    import pyarrow.parquet as pq

    def load_item_map(fname):
        p = find_file(fname, required=False)

        if p is None:
            from huggingface_hub import hf_hub_download
            p = hf_hub_download(
                repo_id="yandex/yambda",
                repo_type="dataset",
                filename=fname,
                local_dir="data",
            )

        m = pq.read_table(p).to_pandas()
        other = [c for c in m.columns if c != "item_id"]

        if len(other) != 1:
            raise ValueError(
                f"Неожиданная структура {fname}: {list(m.columns)}"
            )

        return (
            m.drop_duplicates("item_id")
             .set_index("item_id")[other[0]]
        )

    for col, fname in [
        ("artist_id", "artist_item_mapping.parquet"),
        ("album_id", "album_item_mapping.parquet"),
    ]:
        mapping = load_item_map(fname)

        s = df["item_id"].map(mapping).astype("float64")

        df[col] = np.where(
            s.isna(),
            -1 - df["item_id"].astype(np.int64),
            s,
        ).astype(np.int64)

print("Колонки после восстановления:")
print(list(df.columns))

print(
    "artist_id:", df["artist_id"].nunique(),
    "| album_id:", df["album_id"].nunique()
)

In [ ]:
# Финальный запуск.
# Ничего из Transformer/агентов здесь не запускается.
#res = run_negatives_experiment(
 #   df,
  #  tr.to_numpy(),
   # (va & is_sel).to_numpy(),
    #(va & ~is_sel).to_numpy(),
   # n_neg=20,
#)
#display(res)


## 3. Автоматический поиск признаков (AutoFE)

Основной финальный эксперимент: генерация кандидатов, калибровка и проверка на независимой report-части.

In [ ]:
# =====================================================================
# autofe_search.py  --  ночной эксперимент: ПОИСК ПРИЗНАКОВ-ПРОГРАММ с калибровкой по нулевому распределению
#
# Постановка: задача с непрослушанными кандидатами (негативы, n_neg=20), те же кандидаты и разбиение,
# что в negatives_eval (результаты сопоставимы с твоим запуском: manual 0,7557 / pool 0,7798 / rfe18 0,7772).
#
# Идея (в духе OpenFE [2], но с контролем ложных открытий):
#  1) Новые «примитивы» истории, которых нет в пуле окон: давность, возраст пары, средний и последний
#     интервал между прослушиваниями, накопленное число (по ключам user/item/ui/uart/ualb/art/alb).
#  2) Язык признаков-программ: выражение = op(a, b) над базовыми признаками, глубина до 2.
#     Утечки невозможны по построению (замкнутость): любая построчная функция от признаков,
#     посчитанных по прошлому, сама посчитана по прошлому.
#  3) Оценка кандидата без полного переобучения: бустинг остатка. Базовая модель даёт out-of-fold
#     оценку init_score; на ней обучается маленькая модель только на одном кандидате; прирост NDCG@10
#     на sel - «ценность» кандидата. Шкала: стадия 1 (30% групп, 30 деревьев) -> стадия 2 (все данные).
#  4) НУЛЕВАЯ КАЛИБРОВКА: те же процедуры запускаются на кандидатах с перемешанными значениями
#     (сигнала нет по построению). Порог принятия = 99-й процентиль нулевых приростов; считается доля
#     ожидаемых ложных открытий. Это ответ на «проклятие победителя» из раздела 6.7 отчёта.
#  5) Сравнение на rep при равном числе признаков: ручной / полный пул / RFE из пула / RFE из пула+найденное;
#     абляция «что дало прирост»: примитивы, операторы по категориям, слепой поиск против структурного.
#
# Запуск (в ноутбуке 4 после ячейки с negatives_eval, где уже есть df, tr, va, is_sel):
#   from autofe_search import run_overnight
#   out = run_overnight(df, tr.to_numpy(), (va & is_sel).to_numpy(), (va & ~is_sel).to_numpy(),
#                       out_dir="/kaggle/working/ovn", budget_hours=7.0)
# Перезапуск безопасен: кэши (признаки, базовые оценки, стадия 1) подхватываются.
# =====================================================================
import os, time, json, gc, math, warnings, traceback
import numpy as np, pandas as pd, lightgbm as lgb
from joblib import Parallel, delayed

warnings.filterwarnings("ignore", message="Overriding the parameters")
_NEEDED = ["build_split", "group_metrics", "boot_ci", "LGB", "manual_cols", "FAMILIES", "TAG", "DAY", "SHIFT"]
try:
    from negatives_eval import build_split, group_metrics, boot_ci, LGB, manual_cols, FAMILIES, TAG, DAY, SHIFT
except ImportError:                                               # код вставлен в ноутбук: берём из его глобалов
    import __main__ as _m
    for _n in _NEEDED:
        globals()[_n] = getattr(_m, _n)

CFG = dict(
    n_neg=20, seed=0, budget_hours=7.0, n_jobs=max(1, (os.cpu_count() or 4)),
    s1_frac=0.30, s1_rounds=30, s1_leaves=15, s1_lr=0.10,
    s2_top=250, s2_rounds=100,
    n_null1=300, n_null2=60, null_q=0.99,
    n_blind=6000, blind_base_top=40,
    r2_top=15, r2_base=30,
    Ns=(5, 10, 20, 40), seeds=(0, 1, 2), K_rfe=18, abl_seeds=(0, 1), abl_top=10,
    share_s1=0.30, share_r2=0.10, share_s2=0.12,
    cache_dir=None, min_leaf=200, min_left_rfe=5400, min_left_abl=3600, div_thr=0.9,
)


# ---------------------------------------------------------------------
# 0. Журнал
# ---------------------------------------------------------------------
class Log:
    def __init__(self, path):
        self.path, self.t0 = path, time.time()
    def __call__(self, *a):
        s = f"[{(time.time()-self.t0)/60:7.1f} мин] " + " ".join(str(x) for x in a)
        print(s, flush=True)
        with open(self.path, "a", encoding="utf-8") as f: f.write(s + "\n")


# ---------------------------------------------------------------------
# 1. Новые примитивы истории (по ключу): давность, возраст, интервалы, накопленное число
# ---------------------------------------------------------------------
def query_gap_feats(ev, q, keys, name, snap=DAY, shift=SHIFT):
    """Все значения по событиям СТРОГО до начала суток запроса (весь прошлый период, без окон)."""
    n = len(ev)
    codes = pd.concat([ev[keys], q[keys]], ignore_index=True).groupby(keys, sort=False).ngroup().to_numpy(np.int64)
    ce, cq = codes[:n], codes[n:]
    comb_s = np.sort((ce << shift) | ev["timestamp"].to_numpy(np.int64))
    tq = (q["timestamp"].to_numpy(np.int64) // snap) * snap
    hi = np.searchsorted(comb_s, (cq << shift) | tq, side="left")
    gs = np.searchsorted(comb_s, cq << shift, side="left")
    nb = hi - gs; M = (1 << shift) - 1
    last = comb_s[np.maximum(hi - 1, 0)] & M
    prev = comb_s[np.maximum(hi - 2, 0)] & M
    first = comb_s[np.minimum(gs, n - 1)] & M
    h1, h2 = nb >= 1, nb >= 2
    f32 = lambda x: x.astype(np.float32)
    return {f"{name}_rec_d": f32(np.where(h1, (tq - last) / snap, np.nan)),
            f"{name}_age_d": f32(np.where(h1, (tq - first) / snap, np.nan)),
            f"{name}_gapmean_d": f32(np.where(h2, (last - first) / snap / np.maximum(nb - 1, 1), np.nan)),
            f"{name}_gaplast_d": f32(np.where(h2, (last - prev) / snap, np.nan)),
            f"{name}_ncum": f32(nb)}


KEYS = {"user": ["uid"], "item": ["item_id"], "ui": ["uid", "item_id"], "uart": ["uid", "artist_id"],
        "ualb": ["uid", "album_id"], "art": ["artist_id"], "alb": ["album_id"]}


def build_primitives(ev, q):
    out = {}
    for fam, keys in KEYS.items():
        out.update(query_gap_feats(ev, q, keys, fam))
    return pd.DataFrame(out, index=q.index)


# ---------------------------------------------------------------------
# 2. Язык программ: выражения op(a, b), a/b - имя базового признака или вложенное выражение
# ---------------------------------------------------------------------
def expr_name(s):
    if isinstance(s, str): return s
    if s[0] == "null": return f"NULL{s[1]}[{expr_name(s[2])}]"
    sym = {"div": "/", "ratio1": "/(1+", "diff": "-", "sum": "+", "mul": "*"}[s[0]]
    a, b = expr_name(s[1]), expr_name(s[2])
    return f"({a}{sym}{b}))" if s[0] == "ratio1" else f"({a}{sym}{b})"


def depth(s):
    if isinstance(s, str): return 0
    return 1 + max(depth(s[1]), depth(s[2])) if s[0] != "null" else depth(s[2])


class Getter:
    def __init__(self, X, ci, salt): self.X, self.ci, self.salt = X, ci, salt
    def __call__(self, n): return self.X[:, self.ci[n]]


def eval_expr(s, get):
    if isinstance(s, str): return get(s)
    if s[0] == "null":
        c = eval_expr(s[2], get)
        return np.random.default_rng(s[1] * 2 + get.salt).permutation(c)
    a, b = eval_expr(s[1], get), eval_expr(s[2], get)
    with np.errstate(all="ignore"):
        if s[0] == "div": r = np.where(np.abs(b) < 1e-9, np.nan, a / b)
        elif s[0] == "ratio1": r = a / (b + 1.0)
        elif s[0] == "diff": r = a - b
        elif s[0] == "sum": r = a + b
        elif s[0] == "mul": r = a * b
        else: raise ValueError(s[0])
    r = np.asarray(r, np.float32)
    r[~np.isfinite(r)] = np.nan
    return r


def _ws(cols, fam, kind):
    return sorted(int(c.split("_")[-1][:-1]) for c in cols if c.startswith(f"{fam}_{kind}_") and c.endswith("d") and c.split("_")[-1][:-1].isdigit())


def gen_structured(cols):
    """Структурные кандидаты четырёх категорий (trend, hier, timing, burst). Возвращает список (спецификация, категория)."""
    cs = set(cols); C = []
    op_for = lambda kind: "ratio1" if kind == "cnt" else "diff"
    for fam in FAMILIES:                                              # trend: короткое окно против длинного
        for kind in ("cnt", "rate", "mpr"):
            ws = _ws(cols, fam, kind)
            for i in range(len(ws)):
                for j in range(i + 1, len(ws)):
                    C.append(((op_for(kind), f"{fam}_{kind}_{ws[i]}d", f"{fam}_{kind}_{ws[j]}d"), "trend"))
    pairs = [("ui", "uart"), ("ui", "ualb"), ("ui", "user"), ("uart", "user"), ("ualb", "user"), ("ui", "item"),
             ("item", "art"), ("item", "alb"), ("uart", "art"), ("ualb", "alb"), ("user", "item")]
    for a, b in pairs:                                                # hier: «пара против общего уровня»
        for kind in ("cnt", "rate", "mpr"):
            for w in sorted(set(_ws(cols, a, kind)) & set(_ws(cols, b, kind))):
                C.append(((op_for(kind), f"{a}_{kind}_{w}d", f"{b}_{kind}_{w}d"), "hier"))
    for fam in FAMILIES:                                              # timing: давность относительно привычного интервала
        r, age, gm, gl, nc = (f"{fam}_{s}" for s in ("rec_d", "age_d", "gapmean_d", "gaplast_d", "ncum"))
        if all(x in cs for x in (r, age, gm, gl, nc)):
            C += [(("div", r, gm), "timing"), (("diff", r, gm), "timing"), (("div", r, gl), "timing"), (("div", r, age), "timing"),
                  (("div", gl, gm), "timing"), (("div", nc, age), "timing")]
    for a, b in [("ui", "uart"), ("ui", "user"), ("uart", "user"), ("ui", "ualb")]:
        if f"{a}_rec_d" in cs and f"{b}_rec_d" in cs:
            C += [(("div", f"{a}_rec_d", f"{b}_rec_d"), "timing"), (("diff", f"{a}_rec_d", f"{b}_rec_d"), "timing")]
    for fam in FAMILIES:                                              # burst: доля прослушиваний окна в накопленном
        if f"{fam}_ncum" in cs:
            for w in _ws(cols, fam, "cnt"):
                C.append((("ratio1", f"{fam}_cnt_{w}d", f"{fam}_ncum"), "burst"))
    return C


def gen_blind(top_cols, n, rng, seen):
    ops = ["diff", "mul", "div", "sum"]; out = []; tries = 0
    while len(out) < n and tries < 30 * n:
        tries += 1
        a, b = rng.choice(len(top_cols), 2, replace=False); op = ops[rng.integers(0, 4)]
        if op in ("sum", "mul") and a > b: a, b = b, a
        s = (op, top_cols[a], top_cols[b]); k = expr_name(s)
        if k in seen: continue
        seen.add(k); out.append((s, "blind"))
    return out


# ---------------------------------------------------------------------
# 3. Быстрая метрика и вспомогательные функции обучения
# ---------------------------------------------------------------------
class FastNDCG:
    """NDCG@k по группам (строки отсортированы по g); считает только подходящие группы (elig, есть позитив)."""
    def __init__(self, g, y, elig, k=10):
        self.k, self.y = k, np.asarray(y, np.float64)
        _, self.start, cnt = np.unique(g, return_index=True, return_counts=True)
        self.gid = np.repeat(np.arange(len(cnt)), cnt)
        self.rank_pos = np.arange(len(self.y)) - self.start[self.gid]
        o = np.lexsort((-self.y, self.gid)); r = self.rank_pos
        self.idcg = np.add.reduceat(self.y[o] / np.log2(r + 2) * (r < k), self.start)
        npos = np.add.reduceat(self.y, self.start)
        self.use = (npos > 0) & np.asarray(elig)[self.start]
    def __call__(self, s):
        o = np.lexsort((-np.asarray(s, np.float64), self.gid))
        dcg = np.add.reduceat(self.y[o] / np.log2(self.rank_pos + 2) * (self.rank_pos < self.k), self.start)
        return float((dcg[self.use] / self.idcg[self.use]).mean())


def grp_sizes(g):
    return np.unique(g, return_counts=True)[1]


def fit_np(Xtr, ytr, gtr, Xsel, ysel, gsel, ci, seed=0, rounds=300, params=None):
    p = {**LGB, "seed": seed, **(params or {})}
    dtr = lgb.Dataset(Xtr[:, ci], label=ytr, group=grp_sizes(gtr), params={"max_bin": 63}, free_raw_data=False)
    dsel = lgb.Dataset(Xsel[:, ci], label=ysel, group=grp_sizes(gsel), reference=dtr)
    return lgb.train(p, dtr, rounds, valid_sets=[dsel], callbacks=[lgb.early_stopping(30, verbose=False), lgb.log_evaluation(0)])


def pred_np(m, X, ci):
    return m.predict(X[:, ci], num_iteration=m.best_iteration or None)


def rfe_np(Xtr, ytr, gtr, Xsel, ysel, gsel, ci_names, ci_map, K, drop=0.15, seed=0, rounds=220):
    cols = list(ci_names)
    while len(cols) > K:
        m = fit_np(Xtr, ytr, gtr, Xsel, ysel, gsel, [ci_map[c] for c in cols], seed=seed, rounds=rounds)
        imp = pd.Series(m.feature_importance("gain"), index=cols).sort_values(ascending=False)
        cols = imp.index[:max(K, int(len(cols) * (1 - drop)))].tolist()
    return cols


# ---------------------------------------------------------------------
# 4. Данные: сплиты, кэш признаков, примитивы
# ---------------------------------------------------------------------
def load_or_build(df, masks, cfg, out_dir, log):
    parts, seeds = {}, {"tr": 1, "sel": 2, "rep": 3}
    for nm, mk in zip(("tr", "sel", "rep"), masks):
        cd = cfg.get("cache_dir") or out_dir; os.makedirs(cd, exist_ok=True)
        pr, pf = f"{cd}/rows_{nm}.pkl", f"{cd}/F_{nm}.pkl"
        if os.path.exists(pr) and os.path.exists(pf):
            parts[nm] = (pd.read_pickle(pr), pd.read_pickle(pf)); log(f"[{nm}] из кэша: {len(parts[nm][0])} строк")
            continue
        rows, F = build_split(df, np.asarray(mk), cfg["n_neg"], cfg["seed"] + seeds[nm])
        P = build_primitives(df, rows)
        F = pd.concat([F.reset_index(drop=True), P.reset_index(drop=True)], axis=1)
        rows.to_pickle(pr); F.to_pickle(pf); parts[nm] = (rows, F)
        log(f"[{nm}] строк {len(rows)}, негативов {int(rows['is_neg'].sum())}, признаков {F.shape[1]} (база {F.shape[1]-P.shape[1]} + примитивы {P.shape[1]})")
    return parts


class Ctx:
    pass


def make_ctx(parts, cfg, log):
    c = Ctx()
    (Rtr, Ftr), (Rsel, Fsel), (Rrep, Frep) = parts["tr"], parts["sel"], parts["rep"]
    c.names = list(Ftr.columns); c.ci = {n: i for i, n in enumerate(c.names)}
    c.base_cols = [n for n in c.names if not (n.endswith(("_rec_d", "_age_d", "_gapmean_d", "_gaplast_d", "_ncum")))]
    c.prim_cols = [n for n in c.names if n not in set(c.base_cols)]
    to = lambda F: np.asfortranarray(F.to_numpy(np.float32))
    c.Xtr, c.Xsel, c.Xrep = to(Ftr), to(Fsel), to(Frep)
    c.R = dict(tr=Rtr, sel=Rsel, rep=Rrep)
    c.ytr, c.ysel, c.yrep = (R["label"].to_numpy(np.float32) for R in (Rtr, Rsel, Rrep))
    c.gtr, c.gsel, c.grep = (R["g"].to_numpy() for R in (Rtr, Rsel, Rrep))
    c.elsel, c.elrep = Rsel["elig"].to_numpy(), Rrep["elig"].to_numpy()
    c.nd_sel = FastNDCG(c.gsel, c.ysel, c.elsel)
    log(f"контекст: train {c.Xtr.shape}, sel {c.Xsel.shape}, rep {c.Xrep.shape}; база {len(c.base_cols)} + примитивы {len(c.prim_cols)}")
    return c


def base_oof(c, cfg, out_dir, log):
    """Базовая модель (все базовые признаки + примитивы): out-of-fold оценки на train (2 половины по пользователям)
    и усреднённые оценки на sel/rep. Используются как init_score при оценке кандидатов."""
    path = f"{cfg.get('cache_dir') or out_dir}/base_oof.npz"
    if os.path.exists(path):
        z = np.load(path); log("базовые оценки из кэша"); return z["tr"], z["sel"], z["rep"], json.load(open(f"{out_dir}/base_imp.json"))
    ci = [c.ci[n] for n in c.names]
    users = np.unique(c.gtr // 1000); perm = np.random.default_rng(cfg["seed"]).permutation(users)
    fold_u = {u: i % 2 for i, u in enumerate(perm)}; fold = np.array([fold_u[u] for u in c.gtr // 1000])
    oof = np.zeros(len(c.ytr), np.float32); s_sel, s_rep = 0.0, 0.0; imp = pd.Series(0.0, index=c.names)
    for f in (0, 1):
        t = time.time(); trm = fold != f
        m = fit_np(c.Xtr[trm], c.ytr[trm], c.gtr[trm], c.Xsel, c.ysel, c.gsel, ci, seed=f)
        oof[~trm] = pred_np(m, c.Xtr[~trm], ci); s_sel = s_sel + pred_np(m, c.Xsel, ci) / 2; s_rep = s_rep + pred_np(m, c.Xrep, ci) / 2
        imp += pd.Series(m.feature_importance("gain"), index=c.names)
        log(f"  базовая модель {f+1}/2: {m.best_iteration} деревьев, {round(time.time()-t)} с")
    imp = (imp / imp.sum()).sort_values(ascending=False)
    np.savez(path, tr=oof, sel=s_sel.astype(np.float32), rep=s_rep.astype(np.float32)); json.dump(imp.to_dict(), open(f"{out_dir}/base_imp.json", "w"))
    return oof, s_sel.astype(np.float32), s_rep.astype(np.float32), imp.to_dict()


# ---------------------------------------------------------------------
# 5. Оценка кандидатов бустингом остатка
# ---------------------------------------------------------------------
class Evaluator:
    def __init__(self, c, init_tr, init_sel, rows_mask_tr, rounds, leaves, lr, tag, min_leaf=200):
        self.c, self.rounds, self.tag = c, rounds, tag
        self.idx = np.flatnonzero(rows_mask_tr) if rows_mask_tr is not None else None
        sub = (lambda a: a[self.idx]) if self.idx is not None else (lambda a: a)
        self.Xs = np.asfortranarray(sub(c.Xtr)) if self.idx is not None else c.Xtr
        self.y, self.init, self.sz = sub(c.ytr), sub(init_tr), grp_sizes(sub(c.gtr))
        self.init_sel = init_sel; self.base_nd = c.nd_sel(init_sel)
        self.p = {**LGB, "num_leaves": leaves, "learning_rate": lr, "num_threads": 1, "min_data_in_leaf": min_leaf, "feature_fraction": 1.0,
                  "bagging_fraction": 1.0, "bagging_freq": 0, "verbosity": -1, "metric": "None"}
        self.get_tr, self.get_sel = Getter(self.Xs, c.ci, 0), Getter(c.Xsel, c.ci, 1)

    def gain(self, spec):
        t = time.time()
        try:
            xt, xs = eval_expr(spec, self.get_tr), eval_expr(spec, self.get_sel)
            d = lgb.Dataset(xt[:, None], label=self.y, group=self.sz, init_score=self.init, params={"max_bin": 63, "min_data_in_leaf": self.p["min_data_in_leaf"], "verbosity": -1})
            m = lgb.train(self.p, d, self.rounds)
            g = self.c.nd_sel(self.init_sel + m.predict(xs[:, None])) - self.base_nd
        except Exception as e:
            g = np.nan
            if not getattr(Evaluator, '_warned', False):
                Evaluator._warned = True; print('ПЕРВАЯ ОШИБКА В ОЦЕНКЕ КАНДИДАТА:', repr(e), flush=True)
        return g, time.time() - t


def run_batch(ev, items, n_jobs):
    res = Parallel(n_jobs=n_jobs, backend="threading")(delayed(ev.gain)(s) for s, _ in items)
    return res


def evaluate_list(ev, items, stage, csv_path, n_jobs, log, chunk=200):
    """Оценка списка (спецификация, категория, мета) с чекпойнтами; уже оценённые читаются из csv."""
    done = pd.read_csv(csv_path) if os.path.exists(csv_path) else pd.DataFrame(columns=["name", "cat", "stage", "gain", "sec", "depth", "spec"])
    seen = set(done.loc[done["stage"] == stage, "name"])
    todo = [(s, cat) for s, cat in items if expr_name(s) not in seen]
    for i in range(0, len(todo), chunk):
        part = todo[i:i + chunk]; t = time.time()
        res = run_batch(ev, part, n_jobs)
        rows = [dict(name=expr_name(s), cat=cat, stage=stage, gain=g, sec=sec, depth=depth(s), spec=json.dumps(s, ensure_ascii=False)) for (s, cat), (g, sec) in zip(part, res)]
        done = pd.concat([done, pd.DataFrame(rows)], ignore_index=True); done.to_csv(csv_path, index=False)
        log(f"  {stage}: оценено {min(i+chunk, len(todo))}/{len(todo)} | {round(time.time()-t)} с на пачку | медиана прироста {np.nanmedian([r['gain'] for r in rows]):+.5f}")
    return done


def to_tuple(x):
    return x if isinstance(x, str) else tuple(to_tuple(y) if isinstance(y, list) else y for y in x)


def to_spec(js):
    v = json.loads(js)
    def rec(a): return a if isinstance(a, str) else (a[0], a[1], rec(a[2])) if a[0] == "null" else (a[0], rec(a[1]), rec(a[2]))
    return rec(v)


# ---------------------------------------------------------------------
# 6. Главная процедура
# ---------------------------------------------------------------------
def run_overnight(df, mask_tr, mask_sel, mask_rep, out_dir="results/ovn", **kw):
    cfg = {**CFG, **kw}; os.makedirs(out_dir, exist_ok=True)
    log = Log(f"{out_dir}/log.txt"); T0 = time.time(); deadline = T0 + cfg["budget_hours"] * 3600
    left = lambda: deadline - time.time()
    ptimes = {}
    json.dump({k: (list(v) if isinstance(v, tuple) else v) for k, v in cfg.items()}, open(f"{out_dir}/config.json", "w"), ensure_ascii=False, indent=1)
    df = df[["uid", "timestamp", "item_id", "artist_id", "album_id", "label", "pr"]].reset_index(drop=True)
    log(f"старт; бюджет {cfg['budget_hours']} ч, потоков {cfg['n_jobs']}")

    t = time.time(); parts = load_or_build(df, (mask_tr, mask_sel, mask_rep), cfg, out_dir, log); ptimes["признаки"] = time.time() - t
    c = make_ctx(parts, cfg, log); del parts; gc.collect()
    t = time.time(); init_tr, init_sel, init_rep, imp = base_oof(c, cfg, out_dir, log); ptimes["базовая_модель"] = time.time() - t
    base_cols_all = c.base_cols + c.prim_cols
    log(f"базовая NDCG@10 на sel (быстрая метрика, все группы): {c.nd_sel(init_sel):.4f}")

    # ---- Стадия 1: структурные + слепые кандидаты + нули ----
    t1 = time.time()
    rng = np.random.default_rng(cfg["seed"] + 7)
    rs = np.random.default_rng(cfg["seed"] + 11)
    G = len(np.unique(c.gtr)); gid = np.unique(c.gtr, return_inverse=True)[1]
    sub_groups = rs.random(G) < cfg["s1_frac"]; ev1 = Evaluator(c, init_tr, init_sel, sub_groups[gid], cfg["s1_rounds"], cfg["s1_leaves"], cfg["s1_lr"], "s1", cfg["min_leaf"])
    struct = gen_structured(base_cols_all); seen = {expr_name(s) for s, _ in struct}
    top_base = [n for n in imp if n in c.ci][:cfg["blind_base_top"]]
    log(f"структурных кандидатов: {len(struct)}; верхних базовых признаков для слепого поиска: {len(top_base)}")
    # калибровка скорости
    csv1 = f"{out_dir}/stage1.csv"
    cal = [s for s, _ in struct[:max(8, cfg["n_jobs"] * 2)]]; tc = time.time(); run_batch(ev1, [(s, "") for s in cal], cfg["n_jobs"]); per = (time.time() - tc) / len(cal)
    avail = cfg["share_s1"] * cfg["budget_hours"] * 3600 - (time.time() - t1)
    n_total = int(max(0, avail) / max(per, 1e-3))
    n_blind = int(min(cfg["n_blind"], max(0, n_total - len(struct) - cfg["n_null1"])))
    n_null1 = int(min(cfg["n_null1"], max(20, n_total // 8)))
    log(f"скорость {per:.2f} с/кандидат (с учётом потоков); бюджет стадии 1 -> {n_total} оценок: структурных {len(struct)}, слепых {n_blind}, нулей {n_null1}")
    blind = gen_blind(top_base, n_blind, rng, seen)
    pool_items = struct + blind
    nulls = [(("null", int(k), pool_items[int(i)][0]), "null") for k, i in enumerate(rng.choice(len(pool_items), n_null1, replace=len(pool_items) < n_null1))]
    d1 = evaluate_list(ev1, pool_items + nulls, "s1", csv1, cfg["n_jobs"], log)
    ptimes["стадия1"] = time.time() - t1

    s1 = d1[d1["stage"] == "s1"]; null1 = s1.loc[s1["cat"] == "null", "gain"].dropna(); real1 = s1[s1["cat"] != "null"].dropna(subset=["gain"])
    th1 = float(np.quantile(null1, cfg["null_q"])) if len(null1) >= 10 else float(real1["gain"].quantile(0.99))
    log(f"стадия 1: нулей {len(null1)}, порог (q{cfg['null_q']}) = {th1:+.5f}; реальных выше порога {int((real1['gain']>th1).sum())} из {len(real1)} "
        f"(ожидалось бы ложных ~{(1-cfg['null_q'])*len(real1):.1f})")

    # ---- Раунд 2 (глубина 2): лучшие найденные x верхние базовые ----
    t2 = time.time()
    if left() > cfg.get('min_left_r2', 1800):
        best1 = real1.sort_values("gain", ascending=False).head(cfg["r2_top"])
        base2 = top_base[:cfg["r2_base"]]; r2 = []; seen2 = set(s1["name"])
        for js in best1["spec"]:
            sp = to_spec(js)
            for b in base2:
                for op in ("mul", "div", "diff"):
                    s = (op, sp, b); k = expr_name(s)
                    if k not in seen2 and depth(s) <= 2: seen2.add(k); r2.append((s, "depth2"))
        cap = int(max(0, cfg["share_r2"] * cfg["budget_hours"] * 3600 / max(per, 1e-3)))
        r2 = r2[:cap]; log(f"раунд 2: {len(r2)} кандидатов глубины 2")
        d1 = evaluate_list(ev1, r2, "s1", csv1, cfg["n_jobs"], log)
    ptimes["раунд2"] = time.time() - t2
    s1 = d1[d1["stage"] == "s1"]; real1 = s1[s1["cat"] != "null"].dropna(subset=["gain"])

    # ---- Стадия 2: все данные, больше деревьев, свои нули ----
    t3 = time.time()
    ev2 = Evaluator(c, init_tr, init_sel, None, cfg["s2_rounds"], cfg["s1_leaves"], 0.1, "s2", cfg["min_leaf"])
    top = real1.sort_values("gain", ascending=False).head(cfg["s2_top"]); items2 = [(to_spec(js), cat) for js, cat in zip(top["spec"], top["cat"])]
    nul2 = [(("null", 1000 + k, items2[int(i)][0]), "null") for k, i in enumerate(rng.choice(len(items2), cfg["n_null2"], replace=len(items2) < cfg["n_null2"]))]
    csv2 = f"{out_dir}/stage2.csv"
    d2 = evaluate_list(ev2, items2 + nul2, "s2", csv2, min(cfg["n_jobs"], 4), log, chunk=50)
    s2 = d2[d2["stage"] == "s2"]; null2 = s2.loc[s2["cat"] == "null", "gain"].dropna(); real2 = s2[s2["cat"] != "null"].dropna(subset=["gain"]).sort_values("gain", ascending=False)
    th2 = float(np.quantile(null2, cfg["null_q"])) if len(null2) >= 10 else float(real2["gain"].quantile(0.9))
    surv = real2[real2["gain"] > th2]
    log(f"стадия 2: порог {th2:+.5f}; прошли {len(surv)} из {len(real2)}; лучший прирост {real2['gain'].max():+.5f}")
    ptimes["стадия2"] = time.time() - t3
    pd.concat([real1.assign(stage="s1"), real2.assign(stage="s2")]).to_csv(f"{out_dir}/candidates_all.csv", index=False)
    pool_for_sets = surv if len(surv) >= 3 else real2.head(max(cfg["Ns"]))              # если порог никто не прошёл - берём лучших и помечаем
    flag_no_survivors = len(surv) < 3
    del ev1, ev2; gc.collect()

    # ---- Стадия 3: диверсификация и финальные сравнения ----
    t4 = time.time()
    try:
        res = final_stage(c, cfg, out_dir, log, pool_for_sets, flag_no_survivors, left, th1, th2, null1, null2, real1, real2)
    except Exception:
        log("ОШИБКА на финальной стадии:\n" + traceback.format_exc()); res = {}
    ptimes["финал"] = time.time() - t4; ptimes["всего"] = time.time() - T0
    json.dump(ptimes, open(f"{out_dir}/phase_times.json", "w"), ensure_ascii=False, indent=1)
    try:
        make_report(out_dir, cfg, log)
    except Exception:
        log("ОШИБКА отчёта:\n" + traceback.format_exc())
    log(f"готово за {ptimes['всего']/3600:.2f} ч")
    return res


def diversify(c, specs_df, n_max, sample=150_000, thr=0.9):
    """Отбор разнообразных признаков: пропускаем те, у кого |Спирмен| > thr с уже принятым."""
    rng = np.random.default_rng(0); idx = np.sort(rng.choice(len(c.ytr), min(sample, len(c.ytr)), replace=False))
    get = Getter(np.asfortranarray(c.Xtr[idx]), c.ci, 0); chosen, cols = [], []
    for js, name in zip(specs_df["spec"], specs_df["name"]):
        v = pd.Series(eval_expr(to_spec(js), get)).rank(pct=True).fillna(-1).to_numpy()
        if all(abs(np.corrcoef(v, w)[0, 1]) <= thr for w in cols):
            chosen.append((js, name)); cols.append(v)
        if len(chosen) >= n_max: break
    return chosen


def add_features(c, chosen, tag):
    """Считает выбранные программы на train/sel/rep и возвращает расширенные матрицы и имена."""
    names, cx = [], []
    for js, name in chosen:
        sp = to_spec(js); names.append(name)
        cx.append([eval_expr(sp, Getter(X, c.ci, 0)).astype(np.float32) for X in (c.Xtr, c.Xsel, c.Xrep)])
    ext = lambda X, k: np.asfortranarray(np.column_stack([X] + [col[k] for col in cx])) if cx else X
    return ext(c.Xtr, 0), ext(c.Xsel, 1), ext(c.Xrep, 2), names


def final_stage(c, cfg, out_dir, log, surv, no_surv, left, th1, th2, null1, null2, real1, real2):
    rep_el = c.elrep; ysel, yrep = c.ysel, c.yrep
    metrics = lambda sc: group_metrics(c.grep[rep_el], sc[rep_el], yrep[rep_el])
    Nmax = max(cfg["Ns"]); div = diversify(c, surv, Nmax, thr=cfg['div_thr'])
    log(f"после диверсификации: {len(div)} признаков из {len(surv)}" + (" (ВНИМАНИЕ: порог стадии 2 никто не прошёл - это лучшие по приросту, но не доказанные)" if no_surv else ""))
    pd.DataFrame(div, columns=["spec", "name"]).to_csv(f"{out_dir}/diversified.csv", index=False)
    Xtr, Xsel, Xrep, dnames = add_features(c, div, "all")
    ci = dict(c.ci); 
    for i, n in enumerate(dnames): ci[n] = c.Xtr.shape[1] + i
    base90 = [n for n in c.base_cols]; prim = c.prim_cols; man = manual_cols(c.base_cols)

    def run_set(name, cols, seeds, rounds=300):
        idx = [ci[x] for x in cols]; per = []; t = time.time()
        for sd in seeds:
            m = fit_np(Xtr, c.ytr, c.gtr, Xsel, ysel, c.gsel, idx, seed=sd, rounds=rounds)
            per.append((pred_np(m, Xsel, idx), pred_np(m, Xrep, idx), m.best_iteration))
        ssel = np.mean([p[0] for p in per], 0); srep = np.mean([p[1] for p in per], 0)
        nd_rep = [group_metrics(c.grep[rep_el], p[1][rep_el], yrep[rep_el])["ndcg"].mean() for p in per]
        out = dict(набор=name, признаков=len(cols), sel=c.nd_sel(ssel), rep_ndcg=float(np.mean(nd_rep)), std=float(np.std(nd_rep, ddof=1)) if len(nd_rep) > 1 else np.nan,
                   сек_на_запуск=round((time.time() - t) / len(seeds)), деревьев=int(np.mean([p[2] for p in per])))
        log(f"  [{name}] {len(cols)} приз. | sel {out['sel']:.4f} | rep {out['rep_ndcg']:.4f} ± {out['std']:.4f}")
        return out, metrics(srep)

    results, mets = [], {}
    def add(name, cols, seeds):
        r, m = run_set(name, cols, seeds); results.append(r); mets[name] = m
        pd.DataFrame(results).to_csv(f"{out_dir}/final_sets.csv", index=False)

    add("manual18", man, cfg["seeds"]); add("pool90", base90, cfg["seeds"]); add("pool90+примитивы", base90 + prim, cfg["seeds"])
    # выбор N по sel (seed 0)
    selN = {}
    Ns_eff = sorted({min(N, len(dnames)) for N in cfg["Ns"]} - {0})
    for N in Ns_eff:
        r, _ = run_set(f"выбор N={N}", base90 + prim + dnames[:N], (0,)); selN[N] = r["sel"]
    bestN = max(selN, key=selN.get) if selN else 0
    log(f"лучший N по sel: {bestN} ({selN})")
    cols_best = base90 + prim + dnames[:bestN]
    tag_best = f"pool90+прим+top{bestN}" if bestN else "pool90+примитивы"
    if bestN: add(tag_best, cols_best, cfg["seeds"])
    if left() > cfg["min_left_rfe"]:
        idxm = {n: ci[n] for n in cols_best}
        r90 = rfe_np(Xtr, c.ytr, c.gtr, Xsel, ysel, c.gsel, base90, ci, cfg["K_rfe"]); add(f"RFE-{cfg['K_rfe']} из pool90", r90, cfg["seeds"])
        raug = rfe_np(Xtr, c.ytr, c.gtr, Xsel, ysel, c.gsel, cols_best, ci, cfg["K_rfe"]); add(f"RFE-{cfg['K_rfe']} из {tag_best}", raug, cfg["seeds"])
        json.dump(dict(rfe_pool90=r90, rfe_aug=raug), open(f"{out_dir}/rfe_sets.json", "w"), ensure_ascii=False, indent=1)
        n_new = sum(1 for x in raug if x in dnames); n_prim = sum(1 for x in raug if x in prim)
        log(f"в RFE-{cfg['K_rfe']} из расширенного пула: найденных программ {n_new}, примитивов {n_prim}")
    # абляция по категориям
    if left() > cfg["min_left_abl"]:
        cat_of = surv.set_index("name")["cat"].to_dict()
        for cat in sorted(set(cat_of.values())):
            sub = [(js, nm) for js, nm in zip(surv["spec"], surv["name"]) if cat_of[nm] == cat][:cfg["abl_top"]]
            if not sub: continue
            Xt2, Xs2, Xr2, nm2 = add_features(c, sub, cat); ci2 = dict(c.ci)
            for i, n in enumerate(nm2): ci2[n] = c.Xtr.shape[1] + i
            idx = [ci2[x] for x in base90 + prim + nm2]; per = []
            for sd in cfg["abl_seeds"]:
                m = fit_np(Xt2, c.ytr, c.gtr, Xs2, ysel, c.gsel, idx, seed=sd); per.append(pred_np(m, Xr2, idx))
            nm_ = f"абляция: прим+{len(sub)} из «{cat}»"; mets[nm_] = metrics(np.mean(per, 0))
            results.append(dict(набор=nm_, признаков=len(idx), sel=np.nan, rep_ndcg=float(mets[nm_]["ndcg"].mean()), std=np.nan, сек_на_запуск=np.nan, деревьев=np.nan))
            log(f"  {nm_}: rep {results[-1]['rep_ndcg']:.4f}")
            del Xt2, Xs2, Xr2; gc.collect()
        pd.DataFrame(results).to_csv(f"{out_dir}/final_sets.csv", index=False)
    # разности с доверительными интервалами
    pairs = [("manual18", "pool90"), ("pool90", "pool90+примитивы"), ("pool90+примитивы", tag_best), ("pool90", tag_best), ("manual18", tag_best)]
    nr, na = f"RFE-{cfg['K_rfe']} из pool90", f"RFE-{cfg['K_rfe']} из {tag_best}"
    if nr in mets and na in mets: pairs += [("manual18", nr), (nr, na), ("manual18", na)]
    for k in [k for k in mets if k.startswith("абляция")]: pairs.append(("pool90+примитивы", k))
    diffs = []
    for a, b in pairs:
        for col in ("ndcg", "recall"):
            if a in mets and b in mets:
                d_, ci_ = boot_ci(mets[a], mets[b], col); diffs.append(dict(сравнение=f"{b} − {a}", метрика=col, d=d_, lo=ci_[0], hi=ci_[1]))
    pd.DataFrame(diffs).to_csv(f"{out_dir}/final_diffs.csv", index=False)
    for r in diffs:
        if r["метрика"] == "ndcg": log(f"  {r['сравнение']}: {r['d']:+.4f} [{r['lo']:+.4f}; {r['hi']:+.4f}]")
    json.dump(dict(dnames=dnames, bestN=bestN, th1=th1, th2=th2), open(f"{out_dir}/found.json", "w"), ensure_ascii=False, indent=1)
    return dict(sets=pd.DataFrame(results), diffs=pd.DataFrame(diffs), found=dnames)


# ---------------------------------------------------------------------
# 7. Отчёт: рисунки и markdown
# ---------------------------------------------------------------------
def md(d):
    try: return d.to_markdown(index=False)
    except Exception: return "```\n" + d.to_string(index=False) + "\n```"


def make_report(out_dir, cfg, log):
    import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
    s1 = pd.read_csv(f"{out_dir}/stage1.csv"); s2 = pd.read_csv(f"{out_dir}/stage2.csv") if os.path.exists(f"{out_dir}/stage2.csv") else pd.DataFrame()
    fig, ax = plt.subplots(1, 3, figsize=(17, 4.5))
    real, nul = s1[s1["cat"] != "null"]["gain"].dropna(), s1[s1["cat"] == "null"]["gain"].dropna()
    th = np.quantile(nul, cfg["null_q"]) if len(nul) >= 10 else np.nan
    bins = np.linspace(min(real.min(), nul.min()), max(real.max(), nul.max()), 70)
    ax[0].hist(nul, bins, alpha=.6, label=f"нули (n={len(nul)})", density=True); ax[0].hist(real, bins, alpha=.6, label=f"кандидаты (n={len(real)})", density=True)
    ax[0].axvline(th, color="k", ls="--", label=f"порог q{cfg['null_q']}"); ax[0].set_xlabel("прирост NDCG@10 на sel"); ax[0].set_title("Стадия 1: реальные против нулевых"); ax[0].legend()
    g = s1[s1["cat"] != "null"].dropna(subset=["gain"]).groupby("cat")["gain"].agg(share_pass=lambda x: (x > th).mean(), median="median", n="size")
    ax[1].bar(g.index, g["share_pass"]); ax[1].set_title("Доля кандидатов выше нулевого порога, по категориям")
    for i, (k, r) in enumerate(g.iterrows()): ax[1].text(i, r["share_pass"], f"n={int(r['n'])}", ha="center", va="bottom", fontsize=8)
    if len(s2):
        r2 = s2[s2["cat"] != "null"].dropna(subset=["gain"]); n2 = s2[s2["cat"] == "null"].dropna(subset=["gain"])
        ax[2].scatter(s1.set_index("name").reindex(r2["name"])["gain"], r2["gain"], s=12, alpha=.6, label="кандидаты")
        if len(n2): ax[2].axhline(np.quantile(n2["gain"], cfg["null_q"]), color="k", ls="--", label="порог стадии 2")
        ax[2].set_xlabel("прирост, стадия 1 (30% данных)"); ax[2].set_ylabel("прирост, стадия 2 (все данные)"); ax[2].set_title("Согласованность стадий"); ax[2].legend()
    plt.tight_layout(); plt.savefig(f"{out_dir}/fig_search.png", dpi=130); plt.close()
    if os.path.exists(f"{out_dir}/final_sets.csv"):
        fs = pd.read_csv(f"{out_dir}/final_sets.csv"); fs = fs[~fs["набор"].str.startswith("выбор")]
        fig, ax = plt.subplots(figsize=(9, 0.45 * len(fs) + 1.5)); ax.barh(fs["набор"], fs["rep_ndcg"], xerr=fs["std"].fillna(0), color="#2a6fb0")
        ax.set_xlim(fs["rep_ndcg"].min() - 0.01, fs["rep_ndcg"].max() + 0.005); ax.set_xlabel("NDCG@10 на rep (среднее по seed ± std)"); ax.invert_yaxis()
        plt.tight_layout(); plt.savefig(f"{out_dir}/fig_final.png", dpi=130); plt.close()
    L = ["# Итоги ночного эксперимента\n", f"Бюджет: {cfg['budget_hours']} ч; конфигурация в config.json.\n"]
    pt = json.load(open(f"{out_dir}/phase_times.json")) if os.path.exists(f"{out_dir}/phase_times.json") else {}
    L.append("## Время по этапам (мин)\n\n" + "\n".join(f"- {k}: {v/60:.1f}" for k, v in pt.items()) + "\n")
    if len(nul): L.append(f"## Нулевая калибровка\n\nСтадия 1: нулей {len(nul)}, порог q{cfg['null_q']} = {th:+.5f}; кандидатов выше порога {(real>th).sum()} из {len(real)} (ложных ожидалось ~{(1-cfg['null_q'])*len(real):.1f}).\n")
    for f_, t_ in [("final_sets.csv", "Наборы признаков (rep)"), ("final_diffs.csv", "Разности с 95% ДИ (кластерный bootstrap по пользователям)"), ("diversified.csv", "Найденные программы (после диверсификации)")]:
        if os.path.exists(f"{out_dir}/{f_}"): L.append(f"## {t_}\n\n" + pd.read_csv(f"{out_dir}/{f_}").round(5).pipe(md) + "\n")
    open(f"{out_dir}/summary.md", "w", encoding="utf-8").write("\n".join(L)); log("summary.md записан")

In [ ]:
out = run_overnight(
    df,
    tr.to_numpy(),
    (va & is_sel).to_numpy(),
    (va & ~is_sel).to_numpy(),
    out_dir="/kaggle/working/ovn",
    budget_hours=8.0,
)